# Etapa 2 · Carga do Sul e interrupções da Copel

Daniel Suzuki Naves (RA 2363062), João Pedro Novais Zanusso (RA 2607743) e Kalil Nege (RA 2409020) · UTFPR

**Carga.** Qual será o maior valor horário da carga do subsistema Sul amanhã? A proposta inicial previa estudar a carga do Paraná. No arquivo do ONS que escolhemos, PR, SC e RS aparecem agregados no subsistema Sul.

**Distribuição.** Em quantos dias voltam a ocorrer interrupções não programadas no mesmo conjunto elétrico da Copel? Preparamos essa segunda base para investigar a recorrência das falhas no Paraná e sua associação com chuva e vento.

O calendário vai de 1º de janeiro de 2022 a 16 de novembro de 2026.

In [1]:
from pathlib import Path
import sys

raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/config.py").is_file())
sys.path.insert(0, str(raiz))

from src import coleta, integracao, qualidade, tratamento

## Fontes e cobertura

Na ANEEL, filtramos a Copel Distribuição pelo CNPJ e removemos registros idênticos. Interrupções e ocorrências emergenciais são apuradas separadamente. Junho de 2025 ficou sem registros de interrupções da Copel nesta extração.

Para o clima, usamos o ERA5 em Curitiba, Londrina, Maringá, Cascavel e Ponta Grossa. Com os cinco pontos completos, calculamos a média simples da temperatura e preservamos a maior chuva e a maior rajada do dia entre eles. Essas medidas descrevem o contexto regional do Paraná.

In [2]:
coleta.baixar_originais()
auditoria = {}
carga = tratamento.preparar_carga(auditoria)
clima = tratamento.preparar_clima(auditoria)
interrupcoes = tratamento.preparar_interrupcoes(auditoria)
ocorrencias = tratamento.preparar_ocorrencias(auditoria)

base = integracao.integrar_fontes(interrupcoes, ocorrencias, carga, clima)
cobertura = qualidade.cobertura_fontes(base).set_index("fonte")
cobertura.rename_axis("Fonte").rename(columns={
    "primeiro_dia": "Início", "ultimo_dia": "Fim",
    "dias_com_dado": "Dias com dados", "dias_ausentes": "Dias sem dados",
})

Originais disponíveis.


,Início,Fim,Dias com dados,Dias sem dados
Fonte,,,,
Interrupções Copel,2022-01-01,2026-09-30,1704,77
Ocorrências Copel,2022-01-01,2026-09-30,1716,65
Carga ONS Sul,2022-01-01,2026-11-14,1779,2
Clima dos cinco pontos,2022-01-01,2026-11-10,1775,6


## Carga de amanhã

Para cada dia de origem D, buscamos o máximo das 24 leituras horárias do ONS no dia seguinte, em MWmed. Se uma hora falta, o alvo fica vazio. Os feriados de PR, SC e RS também entram pela data prevista.

Na linha de 20 de abril, por exemplo, a previsão é para o feriado de 21 de abril. Na linha seguinte, o valor desse alvo já aparece na coluna de carga observada. A temperatura é retrospectiva e só poderá entrar no modelo com a defasagem de publicação.

In [3]:
previsao = integracao.preparar_previsao_carga(base)
abril = previsao.set_index("data_origem").loc["2026-04-19":"2026-04-21"]
colunas = {
    "ons_sul_carga_max_horaria_mwmed": "Máximo de hoje (MWmed)",
    "clima_media_temperature_2m_mean": "Temperatura no PR (°C)",
    "data_alvo": "Dia previsto",
    "feriado_pr_d1": "Feriado no PR (D+1)",
    "carga_max_sul_d1_mwmed": "Máximo de amanhã (MWmed)",
}
abril[list(colunas)].rename(columns=colunas).rename_axis("Dia de origem")

,Máximo de hoje (MWmed),Temperatura no PR (°C),Dia previsto,Feriado no PR (D+1),Máximo de amanhã (MWmed)
Dia de origem,,,,,
2026-04-19,15245.948,21.54,2026-04-20,False,17296.640
2026-04-20,17296.640,21.06,2026-04-21,True,17034.196
2026-04-21,17034.196,19.92,2026-04-22,False,18650.321


## Interrupções no mesmo conjunto

Cada linha reúne as interrupções iniciadas em um conjunto elétrico naquele dia. Contamos os identificadores distintos classificados como não programados. Se o conjunto teve apenas desligamentos programados, a contagem é zero. Quando falta qualquer registro, o par conjunto–dia fica ausente.

Veja os dias 7 e 8 de junho no conjunto 14593. A coluna de registros permite distinguir um zero observado de uma lacuna. O exemplo usa um único ano porque a ANEEL mudou o esquema dos identificadores em 2026.

In [4]:
conjuntos, sem_conjunto = integracao.preparar_conjuntos(base)
exemplo = conjuntos.loc[conjuntos.conjunto.eq(14593)].set_index("data").sort_index()
ano = exemplo.loc["2026"]
dias_com_falha = ano.interrupcoes_nao_programadas.gt(0).sum()
print(f"Conjunto 14593 em 2026: {dias_com_falha} dias com falha em {len(ano)} dias com registro.")
print(f"Em todo o período, {sem_conjunto.sum()} registros não programados ficaram sem conjunto válido.")

colunas = {
    "registros_total": "Registros de todos os tipos",
    "interrupcoes_nao_programadas": "Interrupções não programadas",
    "clima_max_precipitacao_mm": "Maior chuva (mm)",
    "clima_max_rajada_kmh": "Maior rajada (km/h)",
}
ano.loc["2026-06-06":"2026-06-09", list(colunas)].rename(columns=colunas).rename_axis("Dia")

Conjunto 14593 em 2026: 271 dias com falha em 273 dias com registro.
Em todo o período, 47 registros não programados ficaram sem conjunto válido.


,Registros de todos os tipos,Interrupções não programadas,Maior chuva (mm),Maior rajada (km/h)
Dia,,,,
2026-06-06,6,6,0.0,25.9
2026-06-07,1,0,0.0,23.4
2026-06-08,2,0,0.3,34.6
2026-06-09,5,4,1.8,50.0


## Bases para a próxima etapa

Antes de exportar, conferimos os alvos D+1 com as leituras horárias e as contagens por conjunto com os microdados da Copel. Os dois Parquets e o manifesto dos originais ficam em `data/processed/`.

In [5]:
qualidade.registrar_manifesto()
qualidade.verificar_base(base)
qualidade.verificar_entregas(base, previsao, conjuntos)
integracao.exportar_entregas(base, previsao, conjuntos)

,Arquivo,Uma linha por,Linhas
0,carga_sul_d1.parquet,Dia de origem,1781
1,interrupcoes_nao_programadas_conjunto_dia.parquet,Conjunto e dia observado,254955
